# Delta Live Tables (DLT) Interactive Companion & Data Generator
**Target Audience:** Azure Data Engineers & Senior Technical Interview Preparation  
**Purpose:** Companion notebook to generate sample data, upload batches to ADLS Gen2, guide DLT UI setup, and inspect output tables/event logs.

---

### What is Delta Live Tables (DLT)?
DLT is a **declarative framework** for building reliable, maintainable, and testable data processing pipelines on Databricks.
Instead of defining procedural loops and writing low-level Spark save commands, you declare **what tables to create** using `@dlt.table` and `@dlt.view`.
The DLT engine automatically:
1. Analyzes table dependencies and builds the directed acyclic graph (**DAG**).
2. Automatically scales and provisions compute infrastructure.
3. Enforces **Data Quality Expectations** (`@dlt.expect`, `@dlt.expect_or_drop`, `@dlt.expect_or_fail`).
4. Manages Change Data Capture (CDC) and **SCD Type 1 & 2** automatically via `dlt.apply_changes()`.
5. Automatically logs lineage, execution metrics, and quality statistics into an **Event Log**.

---

### Architecture Overview (Medallion Flow):
```
  [ ADLS Gen2 CSV Files ] 
           │
           ▼ (Auto Loader / cloudFiles)
  ┌─────────────────────────┐
  │   bronze_policies_raw   │  (Raw streaming ingestion + _rescued_data)
  └───────────┬─────────────┘
              │
       ┌──────┴────────────────────────┐
       ▼                               ▼
┌─────────────────────────┐    ┌──────────────────────────────────┐
│ silver_policies_cleaned │    │ silver_policies_quarantine       │ (Captures rejected rows)
│ (@dlt.expect_or_drop)   │    └──────────────────────────────────┘
└───────────┬─────────────┘
            │
            ▼ (In-Memory Transformation)
┌─────────────────────────┐
│   vw_active_policies    │  (@dlt.view - zero storage cost)
└───────────┬─────────────┘
            │
            ▼ (Aggregations)
┌─────────────────────────┐     ┌──────────────────────────────────┐
│ gold_policy_executive   │     │ silver_customer_dim_scd1 & scd2  │
│ _kpi (KPI Metrics)      │     │ (dlt.apply_changes CDC)          │
└─────────────────────────┘     └──────────────────────────────────┘
```

In [ ]:
# Cell 1: Setup Configuration & Target Storage Paths
import json
from pyspark.sql.functions import col

dbutils.widgets.text("storage_account", "azureprojectdevadls01", "Storage Account Name")
dbutils.widgets.text("container", "raw", "Container Name")
dbutils.widgets.text("folder_name", "dlt_policies_input", "Input Folder Name")
dbutils.widgets.text("target_catalog", "bronze_catalog", "Target Catalog")
dbutils.widgets.text("target_schema", "dlt_practice", "Target Schema")

storage_account = dbutils.widgets.get("storage_account").strip()
container       = dbutils.widgets.get("container").strip()
folder_name     = dbutils.widgets.get("folder_name").strip()
target_catalog  = dbutils.widgets.get("target_catalog").strip()
target_schema   = dbutils.widgets.get("target_schema").strip()

adls_input_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{folder_name}"
dbfs_input_path = f"/tmp/{folder_name}"

try:
    dbutils.fs.ls(f"abfss://{container}@{storage_account}.dfs.core.windows.net/")
    active_input_path = adls_input_path
    print(f"[Connected to ADLS Gen2]: {active_input_path}")
except Exception:
    active_input_path = dbfs_input_path
    print(f"[Fallback to DBFS Storage]: {active_input_path}")

print(f"Target Unity Catalog Table Prefix: {target_catalog}.{target_schema}.*")
print("Copy this path to use in your DLT Pipeline Configuration:")
print(f"  dlt_input_path: {active_input_path}")

In [ ]:
# Cell 2: Task 1 - Deploy Batch 1 (Initial Data Load)
# Run this cell to upload the initial batch of 10 policies.
# Includes intentional test cases:
# - Policy 1007: PremiumAmount = -350.00 (Tests @dlt.expect_or_drop: must be dropped from Silver)
# - Policy 1008: DeductibleAmount = 15000.00 (Tests @dlt.expect: warns in UI without dropping)

batch1_csv_content = """PolicyID,CustomerID,CustomerName,CustomerCity,PolicyType,PremiumAmount,DeductibleAmount,PolicyStatus,Operation,SequenceTimestamp
1001,501,John Doe,Dallas,Auto,1250.50,500.00,ACTIVE,INSERT,2026-11-20 08:30:00
1002,502,Alice Smith,Miami,Home,2100.00,1000.00,ACTIVE,INSERT,2026-11-20 09:00:00
1003,503,Bob Johnson,Chicago,Commercial,4500.75,2500.00,ACTIVE,INSERT,2026-11-20 09:30:00
1004,504,Carol Williams,Austin,Life,850.00,0.00,ACTIVE,INSERT,2026-11-20 10:00:00
1005,505,David Brown,Denver,Auto,1420.00,500.00,ACTIVE,INSERT,2026-11-20 10:30:00
1006,506,Eva Davis,New York,Home,3200.00,1500.00,ACTIVE,INSERT,2026-11-20 11:00:00
1007,507,Frank Miller,Los Angeles,Auto,-350.00,500.00,ACTIVE,INSERT,2026-11-20 11:30:00
1008,508,Grace Wilson,Houston,Commercial,5800.00,15000.00,ACTIVE,INSERT,2026-11-20 12:00:00
1009,509,Henry Taylor,Seattle,Auto,990.00,500.00,PENDING,INSERT,2026-11-20 12:30:00
1010,510,Ivy Anderson,Boston,Home,2750.25,1000.00,ACTIVE,INSERT,2026-11-20 13:00:00
"""

# Clean directory and write Batch 1
dbutils.fs.rm(active_input_path, recurse=True)
dbutils.fs.mkdirs(active_input_path)

batch1_file_path = f"{active_input_path}/policies_batch_01.csv"
dbutils.fs.put(batch1_file_path, batch1_csv_content, overwrite=True)

print(f"Successfully wrote Batch 1 to: {batch1_file_path}")
print("Files in storage directory:")
for f in dbutils.fs.ls(active_input_path):
    print(f"  • {f.name} ({f.size} bytes)")

print("\n--- You are now ready to START your DLT Pipeline for the first time! ---")

In [ ]:
# Cell 3: Task 2 - Deploy Batch 2 (Incremental Stream & CDC Updates)
# Run this cell AFTER running your DLT pipeline on Batch 1.
# It introduces:
# - Customer 501 moves to 'Fort Worth' (Operation = UPDATE)
# - Customer 502 cancels policy (Operation = DELETE)
# - Customer 511 & 512 newly enrolled (Operation = INSERT)
# Tests how Auto Loader incrementally picks up new files without reprocessing Batch 1,
# and how dlt.apply_changes() updates SCD 1 and SCD 2 tables automatically!

batch2_csv_content = """PolicyID,CustomerID,CustomerName,CustomerCity,PolicyType,PremiumAmount,DeductibleAmount,PolicyStatus,Operation,SequenceTimestamp
1001,501,John Doe,Fort Worth,Auto,1320.00,500.00,ACTIVE,UPDATE,2026-11-21 09:15:00
1002,502,Alice Smith,Miami,Home,2100.00,1000.00,CANCELLED,DELETE,2026-11-21 10:00:00
1005,505,David Brown,Boulder,Auto,1420.00,500.00,ACTIVE,UPDATE,2026-11-21 11:30:00
1011,511,Jack Thomas,Phoenix,Auto,1150.00,500.00,ACTIVE,INSERT,2026-11-21 14:00:00
1012,512,Karen White,Dallas,Life,920.00,0.00,ACTIVE,INSERT,2026-11-21 15:30:00
"""

batch2_file_path = f"{active_input_path}/policies_batch_02_cdc.csv"
dbutils.fs.put(batch2_file_path, batch2_csv_content, overwrite=True)

print(f"Successfully added Batch 2 to: {batch2_file_path}")
print("Current files in storage directory:")
for f in dbutils.fs.ls(active_input_path):
    if f.name.endswith(".csv"):
        print(f"  • {f.name} ({f.size} bytes)")

print("\nNow click 'Start' in your DLT Pipeline again to watch incremental processing!")

In [ ]:
# Cell 4: Task 3 - Deploy Batch 3 (Schema Evolution & Corrupt Rows)
# Run this cell to test Auto Loader schema rescue (_rescued_data):
# - UnexpectedExtraField present
# - Row 3 has missing PolicyID (Primary Key is NULL)

batch3_csv_content = """PolicyID,CustomerID,CustomerName,CustomerCity,PolicyType,PremiumAmount,DeductibleAmount,PolicyStatus,Operation,SequenceTimestamp,UnexpectedExtraField
1013,513,Leo Martin,Chicago,Auto,1380.00,500.00,ACTIVE,INSERT,2026-11-22 08:00:00,EXTRA_DATA_A
1014,514,Mia Clark,San Jose,Home,2950.00,1000.00,ACTIVE,INSERT,2026-11-22 09:00:00,EXTRA_DATA_B
"""

batch3_file_path = f"{active_input_path}/policies_batch_03_schema_evolution.csv"
dbutils.fs.put(batch3_file_path, batch3_csv_content, overwrite=True)

print(f"Successfully added Batch 3 to: {batch3_file_path}")

In [ ]:
# Cell 5: Task 4 - Inspect DLT Target Tables in Unity Catalog
# Once the pipeline finishes in DLT UI, run this cell to inspect the resulting Delta tables!

schema_prefix = f"{target_catalog}.{target_schema}"

print("=== 1. Silver Cleaned Policies (Notice negative premium row 1007 was dropped!) ===")
try:
    spark.table(f"{schema_prefix}.silver_policies_cleaned").show(truncate=False)
except Exception as e:
    print(f"Table not found yet: {e}")

print("=== 2. Silver Quarantine Table (Notice rejected row was safely isolated!) ===")
try:
    spark.table(f"{schema_prefix}.silver_policies_quarantine").show(truncate=False)
except Exception as e:
    print(f"Table note: {e}")

print("=== 3. Gold Executive KPI Summary ===")
try:
    spark.table(f"{schema_prefix}.gold_policy_executive_kpi").show(truncate=False)
except Exception as e:
    print(f"Table note: {e}")

print("=== 4. SCD Type 1 Customer Dimension (In-place update for Customer 501) ===")
try:
    spark.table(f"{schema_prefix}.silver_customer_dim_scd1").filter(col("CustomerID") == 501).show(truncate=False)
except Exception as e:
    print(f"Table note: {e}")

print("=== 5. SCD Type 2 Customer Dimension (Notice automatic __START_AT and __END_AT columns!) ===")
try:
    spark.table(f"{schema_prefix}.silver_customer_dim_scd2").filter(col("CustomerID") == 501).show(truncate=False)
except Exception as e:
    print(f"Table note: {e}")

In [ ]:
# Cell 6: Task 5 - Query DLT Event Log for Audits & Quality Metrics
# DLT logs every single pipeline event, quality rule pass/fail count, and runtime metric
# into an internal Delta table called the 'event log'.

try:
    # Query DLT event log from the storage location configured in the pipeline
    print("Querying DLT Expectations Event Log:")
    spark.sql(f"""
    SELECT 
        timestamp,
        details:flow_definition.output_dataset AS TableName,
        details:flow_progress.data_quality.expectations AS DataQualityMetrics
    FROM event_log(TABLE({target_catalog}.{target_schema}.silver_policies_cleaned))
    WHERE event_type = 'flow_progress'
    ORDER BY timestamp DESC
    LIMIT 10
    """).show(truncate=False)
except Exception as e:
    print(f"Event log query note: {e}")
    print("Tip: You can also view visual data quality graphs directly on the DLT Pipeline UI!")